# Actual code

Spearman version of [`clinvar_topNvars_scatterplot_master_file.ipynb`](clinvar_topNvars_scatterplot_master_file.ipynb).
The ClinVar side (x-axis: per-gene auROC on pathogenic vs benign) is unchanged. The UKBBGym side
(y-axis) swaps the top-N summary for a **per-gene Spearman correlation** between the tool's
direction-corrected score and the direction-corrected Genebass beta, over *all* rare variants of
the gene rather than only its top N. Same per-gene -> per-tool aggregation as before: mean over
genes, +/- `ci_fold` x SEM.

The correlation follows the same rank-then-`pl.corr` pattern as section 1 of
[`master_file_correlations.ipynb`](../master_file_correlations.ipynb) (Spearman == Pearson on
ranks), computed for every (region, annotation) group at once instead of looping.

**`CLINVAR_PATH` stays a separate file on purpose.** `MASTER_PATH` does have a
`clinical_significance` column, but it isn't a superset of `CLINVAR_PATH` -- it's restricted to
**670 genes**, against `CLINVAR_PATH`'s **17,683**. Pulling ClinVar labels from `MASTER_PATH`
instead would silently confine the whole auROC analysis to the ~4% of genes the master table
happens to cover, for reasons unrelated to ClinVar coverage.


In [ ]:
import sys
from pathlib import Path
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'utils' / 'variant_filtering.py').exists())
sys.path.insert(0, str(REPO_ROOT))
import math
import yaml
import numpy as np
import polars as pl
import polars.selectors as cs
from scipy import stats
from plotnine import *
import matplotlib.pyplot as plt
plt.rcParams['svg.fonttype'] = 'none'

from utils.variant_filtering import (
    load_config, load_variant_class, build_variant_filters, filter_covered,
    scan_variants, pick_annos, gene_trait_tool_correlations, pick_best_trait_per_gene, env_override, fetch_hf_data,
)

# ===================== Parameters (env_override(NAME, default) -- set UKBBGYM_NAME to override) =====================
variant_class       = env_override('VARIANT_CLASS', 'missense')    # defines variant filters + tool set
selected_categories = env_override('SELECTED_CATEGORIES',
                                    ['missense', 'conservation', 'genetic_diversity', 'gnomad'], 'list')

mac                 = env_override('MAC', 20, int)                  # rare-variant cap (phenotype correlations)
only_snps           = env_override('ONLY_SNPS', True, bool)
exclude_clinvar     = env_override('EXCLUDE_CLINVAR', False, bool)
only_clinvar        = env_override('ONLY_CLINVAR', False, bool)

MIN_VARIANTS_PER_GENE = env_override('MIN_VARIANTS', 100, int)   # (gene, tool) pairs with fewer scored variants are dropped from the y-axis
corr_stat  = 'mean'     # 'median' or 'mean' per-gene Spearman rho for the y-axis
auroc_stat = 'mean'     # 'median' or 'mean' ClinVar auROC for the x-axis
ci_fold    = 1.96      # fold for confidence interval (e.g., 1.96 for 95% CI)

# Coverage filter (see "Common predictors, coverage filter" section below): 
COVERAGE_FILTER_ENABLED = True

COVERAGE_EXEMPT_ANNOS = []

# ClinVar pathogenicity labels
patho_labels  = ['Pathogenic', 'Likely_pathogenic']
benign_labels = ['Benign', 'Likely_benign']         # None -> all non-pathogenic treated as benign
MIN_PATHO_PER_GENE    = 5
MIN_BENIGN_PER_GENE   = 5

# ===================== Configs =====================
CFG = str(REPO_ROOT / 'configs')
anno_config_df, all_annotation_list = load_config(CFG, 'config_correlations.yaml')
vc = load_variant_class(CFG, variant_class)

# --- Variant annotations for the target genes ---
_dynamic_filters = build_variant_filters(vc, only_snps=only_snps, only_clinvar=only_clinvar,
                                          exclude_clinvar=exclude_clinvar)

# Tools (annotations) evaluated in BOTH analyses
selected_annos = anno_config_df.filter(pl.col('category').is_in(selected_categories))['annotation'].to_list()

# ===================== Local data paths =====================
MASTER_PATH  = env_override('MASTER_PATH', fetch_hf_data('genebass_annotated.parquet', REPO_ROOT))
CLINVAR_PATH = env_override('CLINVAR_PATH', fetch_hf_data('other_benchmarks/clinvar_annotated.parquet', REPO_ROOT))
FIG_DIR      = env_override('FIG_DIR', '../../../paper_figures')

print(f"variant_class = {variant_class}")
print(f"categories    = {selected_categories}")
print(f"{len(selected_annos)} tools: {selected_annos}")
anno_config_df.filter(pl.col('annotation').is_in(selected_annos))

In [ ]:
# ===================== ClinVar per-gene auROC =====================

def _label_mask(s, labels):
    m = pl.Series([False] * len(s))
    for lbl in labels:
        m = m | s.str.contains(lbl, literal=True).fill_null(False)
    return m

clinvar = (
    pl.scan_parquet(CLINVAR_PATH)
    .filter((pl.col('ref').str.len_chars() == 1) & (pl.col('alt').str.len_chars() == 1))
    .filter(*_dynamic_filters)
)

clin_annos = [a for a in selected_annos if a in clinvar.collect_schema().names()]

# gnomAD AF has no `_is_na` companion -- a null here isn't "not scored", it's "never observed in
gnomad_annos = [a for a in clin_annos
                if a in anno_config_df.filter(pl.col('category') == 'gnomad')['annotation'].to_list()]
clinvar = (clinvar.select(['id', 'region', 'clinical_significance'] + clin_annos)
           .with_columns([pl.col(c).fill_null(0) for c in gnomad_annos])
           .collect())

# Pathogenic / benign masks (handles compound labels like "Benign/Likely_benign")
clin_sig  = clinvar['clinical_significance'].fill_null('')
is_patho  = _label_mask(clin_sig, patho_labels)
is_benign = _label_mask(clin_sig, benign_labels) if benign_labels is not None else ~is_patho
clinvar   = clinvar.with_columns(is_patho=is_patho.cast(pl.Int8)).filter(is_patho | is_benign)
if gnomad_annos:
    print(f'gnomAD AF filled with 0 where null: {gnomad_annos}')

# Long format + direction correction (same melt+direction pattern used for the correlations)
clin_long = (
    clinvar
    .unpivot(index=['id', 'region', 'is_patho'], on=clin_annos,
             variable_name='annotation', value_name='raw')
    .drop_nulls('raw')
    .drop_nans('raw')
    .join(anno_config_df.select(['annotation', 'annotation_dir']), on='annotation', how='left')
    .with_columns(score = pl.col('raw').cast(pl.Float64) * pl.col('annotation_dir').cast(pl.Float64))
)

# Per-gene, per-tool auROC (pathogenic vs benign), with coverage thresholds.
clinvar_auroc_df = (
    clin_long
    .with_columns(rank=pl.col('score').rank('average').over(['region', 'annotation']))
    .group_by(['region', 'annotation'])
    .agg(n_patho=pl.col('is_patho').sum(), n_total=pl.len(),
         rank_sum_patho=(pl.col('rank') * pl.col('is_patho')).sum())
    .with_columns(n_benign=pl.col('n_total') - pl.col('n_patho'))
    .filter(pl.col('n_patho') >= MIN_PATHO_PER_GENE, pl.col('n_benign') >= MIN_BENIGN_PER_GENE)
    .with_columns(auroc=(pl.col('rank_sum_patho') - pl.col('n_patho') * (pl.col('n_patho') + 1) / 2)
                  / (pl.col('n_patho') * pl.col('n_benign')))
    .join(anno_config_df.select(['annotation', 'label', 'color', 'category']), on='annotation', how='left')
)

for row in clinvar_auroc_df.head(5).iter_rows(named=True):     # verify the identity before trusting it
    g = clin_long.filter(pl.col('region') == row['region'], pl.col('annotation') == row['annotation'])
    a = g.filter(pl.col('is_patho') == 1)['score'].to_numpy()
    b = g.filter(pl.col('is_patho') == 0)['score'].to_numpy()
    u = stats.mannwhitneyu(a, b, alternative='greater').statistic / (len(a) * len(b))
    assert abs(u - row['auroc']) < 1e-9, (u, row['auroc'])
print('auROC identity verified against scipy.stats.mannwhitneyu on 5 gene x tool groups')

print(f"ClinVar auROC: {clinvar_auroc_df.height} rows "
      f"({clinvar_auroc_df['region'].n_unique()} genes x {clinvar_auroc_df['annotation'].n_unique()} tools)")

clinvar_auroc_df.sort(['annotation', 'auroc'], descending=[False, True])

In [ ]:
# ===================== Per-gene Spearman rho (tool score vs Genebass beta) =====================
ukbb_lf = scan_variants(MASTER_PATH, vc, only_snps=only_snps, only_clinvar=only_clinvar,
                        exclude_clinvar=exclude_clinvar)
ukbb_annos = pick_annos(anno_config_df, all_annotation_list, selected_categories,
                        ukbb_lf.collect_schema().names())

corr_gene_trait = gene_trait_tool_correlations(ukbb_lf, mac, ukbb_annos, anno_config_df,
                                               selected_categories, MIN_VARIANTS_PER_GENE)

corr_genewise = (
    pick_best_trait_per_gene(corr_gene_trait, group_col='region')
    .rename({'corr_beta': 'spearman'})
)
print(f"corr_genewise: {corr_genewise.shape} "
      f"({corr_genewise['region'].n_unique()} genes x {corr_genewise['annotation'].n_unique()} tools, "
      f">= {MIN_VARIANTS_PER_GENE} variants per gene)")
corr_genewise.sort(['annotation', 'spearman'], descending=[False, True])

## Common predictors, coverage filter

In [ ]:
common_annos = sorted(set(clinvar_auroc_df['annotation'].unique()) & set(corr_genewise['annotation'].unique()))
K = len(common_annos)
print(f'{K} predictors common to both sides: {common_annos}')

# The gene *universe* is defined by `coverage_annos` (common_annos minus COVERAGE_EXEMPT_ANNOS):
# a gene must be scored by every one of those. Rows for an exempt tool (e.g. Pairformer) are kept
# wherever they exist for a covered gene, but the tool doesn't itself gate which genes make the
# cut -- so a sparse newcomer can't collapse the shared gene set for the other tools.
coverage_annos = [a for a in common_annos if a not in COVERAGE_EXEMPT_ANNOS]
exempted = [a for a in common_annos if a in COVERAGE_EXEMPT_ANNOS]
if exempted:
    print(f'exempt from coverage requirement (kept where scored, not required): {exempted}')

clinvar_auroc_df = clinvar_auroc_df.filter(pl.col('annotation').is_in(common_annos))
corr_genewise = corr_genewise.filter(pl.col('annotation').is_in(common_annos))

# COVERAGE_FILTER_ENABLED (set in the Setup cell) toggles the shared coverage filter
if COVERAGE_FILTER_ENABLED:
    clinvar_covered = (clinvar_auroc_df.filter(pl.col('annotation').is_in(coverage_annos))
                        .group_by('region').agg(n=pl.col('annotation').n_unique())
                        .filter(pl.col('n') == len(coverage_annos)).select('region'))
    corr_covered = (corr_genewise.filter(pl.col('annotation').is_in(coverage_annos))
                     .group_by('region').agg(n=pl.col('annotation').n_unique())
                     .filter(pl.col('n') == len(coverage_annos)).select('region'))
    clinvar_auroc_df = clinvar_auroc_df.join(clinvar_covered, on='region', how='semi')
    corr_genewise = corr_genewise.join(corr_covered, on='region', how='semi')

print(f'ClinVar:  {clinvar_auroc_df["region"].n_unique()} genes with all {len(coverage_annos)} core predictors')
print(f'UKBBGym:  {corr_genewise["region"].n_unique()} genes with all {len(coverage_annos)} core predictors')
for a in exempted:
    print(f'  {a}: {clinvar_auroc_df.filter(pl.col("annotation") == a)["region"].n_unique()} of those genes scored '
          f'(ClinVar), {corr_genewise.filter(pl.col("annotation") == a)["region"].n_unique()} (UKBBGym)')

## Scatter: per-gene Spearman rho vs ClinVar auROC, per tool

In [ ]:
# ===================== Spearman rho vs ClinVar auROC -> scatter =====================

# y-axis: per tool, mean (or median) over genes of that gene's Spearman rho
corr_by_tool = (
    corr_genewise
    .group_by('annotation')
    .agg(
        mean_corr   = pl.col('spearman').mean(),
        std_corr    = pl.col('spearman').std(),
        median_corr = pl.col('spearman').median(),
        ci_quantile_low_corr  = pl.col('spearman').quantile(0.025),
        ci_quantile_high_corr = pl.col('spearman').quantile(0.975),
        n_genes_pheno = pl.col('region').n_unique(),
    )
    .with_columns(
        ukbbgym_corr = pl.col(f'{corr_stat}_corr'),
        sem_corr = pl.col('std_corr') / pl.col('n_genes_pheno').sqrt(),
    )
    .with_columns(
        ci_low_corr  = pl.col('mean_corr') - ci_fold * pl.col('sem_corr'),
        ci_high_corr = pl.col('mean_corr') + ci_fold * pl.col('sem_corr'),
    )
)

# x-axis: per tool, mean (or median) per-gene ClinVar auROC
auroc_by_tool = (
    clinvar_auroc_df
    .group_by('annotation')
    .agg(
        mean_auroc   = pl.col('auroc').mean(),
        std_auroc    = pl.col('auroc').std(),
        median_auroc = pl.col('auroc').median(),
        ci_quantile_low_auroc  = pl.col('auroc').quantile(0.25),
        ci_quantile_high_auroc = pl.col('auroc').quantile(0.75),
        n_genes_clinvar = pl.col('region').n_unique()
    )
    .with_columns(
        clinvar_auroc = pl.col(f'{auroc_stat}_auroc'),
        sem_auroc = pl.col('std_auroc') / pl.col('n_genes_clinvar').sqrt()
    )
    .with_columns(
        ci_low_auroc = pl.col('mean_auroc') - ci_fold * pl.col('sem_auroc'),
        ci_high_auroc = pl.col('mean_auroc') + ci_fold * pl.col('sem_auroc'),
    )
)

scatter_df = (
    corr_by_tool
    .join(auroc_by_tool, on='annotation', how='inner')
    .join(anno_config_df.select(['annotation', 'label', 'color', 'category']), on='annotation', how='left')
    .sort('clinvar_auroc', descending=True)
)

cat_color = dict(
    anno_config_df.filter(pl.col('annotation').is_in(scatter_df['annotation']))
    .group_by('category').agg(pl.col('color').first()).iter_rows()
)

# ===================== Across-tool Spearman (one point per predictor) =====================
# Does a tool's ClinVar auROC track its phenotype correlation? Computed over the tool-level
# summary points that are actually drawn, both for all predictors and for the missense VSMs
# alone (the subset re-plotted in the next cell).
MISSENSE_VSM_LABELS = ["popEVE", "CPT-1", "BayesDel", "REVEL", "ClinPred",
                       "AlphaMissense", "ESM1v", "MSA Pairformer"]

N_RESAMPLES = 1_000_000

def tool_level_spearman(df, n_resamples=N_RESAMPLES, seed=0, batch=10_000):
    """Across-tool Spearman rho with a *permutation* P-value.

    The null is built by permuting the pairing between the two axes
    (`permutation_type='pairings'`). scipy enumerates all n! pairings exactly when
    n! <= n_resamples -- true for the 8 missense VSMs (8! = 40,320) -- and samples
    randomly otherwise (17 predictors). Either way the P is not the asymptotic
    t-approximation `stats.spearmanr` returns by default, which is unreliable at
    n = 8-17 (here n counts predictors, not genes).

    The statistic is Pearson-on-ranks -- identical to Spearman, ties included, since
    `rankdata` does the tie correction -- written to broadcast over a whole batch of
    permutations at once. Calling `stats.spearmanr` once per resample instead costs
    ~130 s at a million resamples; this runs in under a second. `batch` caps how many
    permutations are materialised at a time so memory stays flat.
    """
    rank_x = stats.rankdata(df['clinvar_auroc'].to_numpy())
    rank_y = stats.rankdata(df['ukbbgym_corr'].to_numpy())
    y_centred = rank_y - rank_y.mean()
    y_norm = np.sqrt((y_centred ** 2).sum())

    def spearman_stat(a, axis=-1):
        a = np.asarray(a, dtype=float)
        a_centred = a - a.mean(axis=axis, keepdims=True)
        return ((a_centred * y_centred).sum(axis=axis)
                / (np.sqrt((a_centred ** 2).sum(axis=axis)) * y_norm))

    res = stats.permutation_test(
        (rank_x,), spearman_stat, permutation_type='pairings', alternative='two-sided',
        vectorized=True, n_resamples=n_resamples, batch=batch,
        rng=np.random.default_rng(seed),
    )
    return (float(spearman_stat(rank_x)), float(res.pvalue), df.height,
            math.factorial(df.height) <= n_resamples)

rho_all, p_all, n_all, exact_all = tool_level_spearman(scatter_df)
rho_mis, p_mis, n_mis, exact_mis = tool_level_spearman(
    scatter_df.filter(pl.col('label').is_in(MISSENSE_VSM_LABELS)))
print(f'across-tool Spearman, all predictors : rho = {rho_all:.3f}, P = {p_all:.2g}, '
      f'n = {n_all} ({"exact" if exact_all else "randomized"} permutation)')
print(f'across-tool Spearman, missense VSMs  : rho = {rho_mis:.3f}, P = {p_mis:.2g}, '
      f'n = {n_mis} ({"exact" if exact_mis else "randomized"} permutation)')

def fmt_p(pval, n_resamples=N_RESAMPLES):
    """P for the figure: mathtext scientific notation below 1e-3, plain decimal above.

    A randomized permutation P bottoms out at 1/(n_resamples+1); report that as an
    upper bound rather than as a point estimate.
    """
    floor = 1 / (n_resamples + 1)
    prefix, val = ('< ', floor) if pval <= floor else ('', pval)
    if val >= 1e-3:
        return f'{prefix}{val:.2f}'
    mant, exp = f'{val:.1e}'.split('e')
    return f'{prefix}{mant} \u00d7 10$^{{{int(exp)}}}$'

# Stats box under the legend: names right-aligned into `annot_x`, numbers left-aligned just
# after it, so the two rows line up as a two-column block. `$P$` renders italic via mathtext.
annot_names = 'all scores,\nmissense scores,'
annot_stats = (f'$\\rho$ = {rho_all:.2f}, $P$ = {fmt_p(p_all)}\n'
               f'$\\rho$ = {rho_mis:.2f}, $P$ = {fmt_p(p_mis)}')

_x_lo, _x_hi = scatter_df['clinvar_auroc'].min(), scatter_df['clinvar_auroc'].max()
_y_lo, _y_hi = scatter_df['ukbbgym_corr'].min(), scatter_df['ukbbgym_corr'].max()
_x_rng, _y_rng = _x_hi - _x_lo, _y_hi - _y_lo
annot_x = _x_lo + 0.16 * _x_rng     # column split
annot_y = _y_lo + 0.755 * _y_rng    # top of the box, just below the legend

plot = (
    ggplot(scatter_df, aes(x='clinvar_auroc', y='ukbbgym_corr'))
    # + geom_errorbar(aes(ymin='ci_low_corr', ymax='ci_high_corr'), width=0, color='black')
    # + geom_errorbarh(aes(xmin='ci_low_auroc', xmax='ci_high_auroc'), height=0, color='black')
    + geom_point(aes(fill='category'), color='black', size=4.5, stroke=0.6)
    + geom_text(aes(label='label'), color='black', size=11, ha='right', nudge_x=-0.003, va='bottom', nudge_y=0.001)
    + annotate('text', x=annot_x, y=annot_y, label=annot_names,
               ha='right', va='top', size=14, color='black', lineheight=1.5)
    + annotate('text', x=annot_x + 0.012 * _x_rng, y=annot_y, label=annot_stats,
               ha='left', va='top', size=14, color='black', lineheight=1.5)
    + scale_fill_manual(values=cat_color, name='Tool category')
    + labs(
        x=f"{auroc_stat.capitalize()} per-gene auROC on ClinVar pathogenicity\n({clinvar_auroc_df['region'].n_unique()} genes, ≥ {MIN_PATHO_PER_GENE} pathogenic variants)",
        y=f"{corr_stat.capitalize()} per-gene Spearman correlation with the phenotype\n({corr_by_tool['n_genes_pheno'].max()} genes, ≥ {MIN_VARIANTS_PER_GENE} variants)",
    )
    + theme_minimal()
    + theme(
        figure_size=(9, 7.5),
        axis_text=element_text(size=14),
        axis_title=element_text(size=14, lineheight=1.4),
        legend_position=(0.075, 0.95),
        legend_text=element_text(size=14),
        legend_title=element_text(size=14),
        legend_background=element_rect(fill="white", color="white", alpha=1),
        plot_background=element_rect(fill="white", color="white"),
        panel_grid_major=element_line(color="#cccccc", size=0.6),
        panel_grid_minor=element_line(color="#dddddd", size=0.3),
    )
)

plot.save(f"{FIG_DIR}/F6_clinvar_vs_ukbbgym_spearman.svg", dpi=200)

plot

In [ ]:
# ===================== Missense tools only =====================
MISSENSE_ANNOS = MISSENSE_VSM_LABELS   # defined with the across-tool Spearman above

scatter_df_mis = scatter_df.filter(pl.col('label').is_in(MISSENSE_ANNOS))
_mis_x_lo, _mis_x_hi = scatter_df_mis['clinvar_auroc'].min(), scatter_df_mis['clinvar_auroc'].max()
_mis_y_lo, _mis_y_hi = scatter_df_mis['ukbbgym_corr'].min(), scatter_df_mis['ukbbgym_corr'].max()
_mis_x_rng, _mis_y_rng = _mis_x_hi - _mis_x_lo, _mis_y_hi - _mis_y_lo

plot = (
    ggplot(scatter_df_mis, aes(x='clinvar_auroc', y='ukbbgym_corr'))
    + geom_point(aes(fill='category'), color='black', size=4.5, stroke=0.6)
    # + geom_errorbar(aes(ymin='ci_low_corr', ymax='ci_high_corr'), width=0.005, size=0.8, color='black')
    + geom_text(aes(label='label'), color='black', size=11, ha='left', nudge_x=0.001)
    + annotate('text', x=_mis_x_lo + 0.60 * _mis_x_rng, y=_mis_y_lo,
               label='missense scores,', ha='right', va='bottom', size=12, color='black')
    + annotate('text', x=_mis_x_lo + 0.615 * _mis_x_rng, y=_mis_y_lo,
               label=f'$\\rho$ = {rho_mis:.2f}, $P$ = {fmt_p(p_mis)}',
               ha='left', va='bottom', size=12, color='black')
    + scale_fill_manual(values=cat_color, name='Tool category')
    + labs(
        x=f"{auroc_stat.capitalize()} per-gene auROC on ClinVar pathogenicity\n({clinvar_auroc_df['region'].n_unique()} genes, ≥ {MIN_PATHO_PER_GENE} pathogenic variants)",
        y=f"{corr_stat.capitalize()} per-gene Spearman ρ (± {ci_fold}×SEM) with the phenotype\n({corr_by_tool['n_genes_pheno'].max()} genes, ≥ {MIN_VARIANTS_PER_GENE} variants)",
    )
    + theme_minimal()
    + theme(
        figure_size=(9, 7.5),
        axis_text=element_text(size=14),
        axis_title=element_text(size=14, lineheight=1.4),
        legend_position=(0.75, 0.95),
        legend_text=element_text(size=14),
        legend_title=element_text(size=14),
        legend_background=element_rect(fill="white", color="white", alpha=1),
        plot_background=element_rect(fill="white", color="white"),
        panel_grid_major=element_line(color="#cccccc", size=0.6),
        panel_grid_minor=element_line(color="#dddddd", size=0.3),
    )
)

plot.save(f"{FIG_DIR}/F5_clinvar_spearman_missense.svg", dpi=200)

plot
